# Final-evaluation rerun: Random Forest

This notebook reruns the established tuning procedure on the **new frozen development split** created by notebook 10. It never loads the locked final-test rows. It selects validation thresholds for 70%, 75%, 80%, 85%, and 90% target recall, then saves the frozen model and artifacts for notebook 17.

# Random Forest Model Optimisation

Main choices:

1. Hyperparameters are selected using cross-validation AUPRC on the model-training set.
2. The probability threshold is selected only on the validation set.
3. The final test set is used once, after both the model settings and threshold have been fixed.
4. The target validation recall is 80%, matching the current project requirement.


In [1]:
import json
import joblib

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## 1. Load the cleaned dataset and create the output folder

In [3]:
PROJECT_ROOT = Path.cwd()

for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = (
        parent
        / "Processed_Dataset"
        / "diabetic_data_cleaned_stage1.csv"
    )

    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError(
        "Could not find "
        "Processed_Dataset/diabetic_data_cleaned_stage1.csv"
    )

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Model_Results"
    / "random_forest_optimisation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.read_csv(DATA_PATH)

print("Dataset path:")
print(DATA_PATH)

print("\nDataset shape:")
print(df.shape)

df.head()


# Final-evaluation artifacts are kept separate from the earlier development runs.
FINAL_EVALUATION_DIR = PROJECT_ROOT / "Final_Evaluation"
SPLIT_DIR = FINAL_EVALUATION_DIR / "Data_Splits"
OUTPUT_DIR = FINAL_EVALUATION_DIR / "Model_Artifacts" / "random_forest"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nFinal-evaluation output directory:")
print(OUTPUT_DIR)


Dataset path:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Processed_Dataset/diabetic_data_cleaned_stage1.csv

Dataset shape:
(69987, 56)

Final-evaluation output directory:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Final_Evaluation/Model_Artifacts/random_forest


## 2. Select the same modelling features used by the other models

In [4]:
target_col = "readmitted_30"

categorical_features = [
    "gender",
    "race_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "medical_specialty_group",
    "primary_diagnosis",
    "hba1c_group",
    "max_glu_serum",
    "diabetesMed"
]

numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

model_features = (
    categorical_features
    + numeric_features
)

missing_features = [
    feature
    for feature in model_features
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        "These modelling features are missing: "
        f"{missing_features}"
    )

X = df[model_features].copy()
y = df[target_col].astype(int).copy()

print("X shape:")
print(X.shape)

print("\nFeatures used:")
print(X.columns.tolist())

print("\nTarget counts:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True))


X shape:
(69987, 18)

Features used:
['gender', 'race_group', 'age_group', 'admission_source_group', 'discharge_group', 'medical_specialty_group', 'primary_diagnosis', 'hba1c_group', 'max_glu_serum', 'diabetesMed', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Target counts:
readmitted_30
0    63702
1     6285
Name: count, dtype: int64

Target proportions:
readmitted_30
0    0.910198
1    0.089802
Name: proportion, dtype: float64


In [5]:
forbidden_features = {
    "race",
    "age",
    "medical_specialty",
    "diag_1",
    "A1Cresult",
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr"
}

unexpected_features = (
    forbidden_features
    .intersection(X.columns)
)

assert not unexpected_features, (
    "Unexpected or potentially leaking features found: "
    f"{unexpected_features}"
)

assert not X.columns.duplicated().any(), (
    "Duplicate column names were found in X."
)

assert len(X) == len(y), (
    "X and y contain different numbers of rows."
)

assert y.isna().sum() == 0, (
    "The target contains missing values."
)

assert set(y.unique()).issubset({0, 1}), (
    "The target must contain only 0 and 1."
)

print("Feature and target checks passed.")


Feature and target checks passed.


## 3. Create model-training, validation, and final-test sets

The final test set must remain untouched until the model settings and threshold have been selected.

In [6]:

# IMPORTANT: the final-test rows are deliberately NOT loaded in this notebook.
# Notebook 10 creates one fixed split that every model reuses.

SPLIT_DIR = PROJECT_ROOT / "Final_Evaluation" / "Data_Splits"

train_split_path = SPLIT_DIR / "model_train_rows.csv"
validation_split_path = SPLIT_DIR / "threshold_validation_rows.csv"

if not train_split_path.exists() or not validation_split_path.exists():
    raise FileNotFoundError(
        "Final split files are missing. Run 10_create_final_split.ipynb first."
    )

train_idx = (
    pd.read_csv(train_split_path)["row_position"]
    .astype(int)
    .to_numpy()
)
val_idx = (
    pd.read_csv(validation_split_path)["row_position"]
    .astype(int)
    .to_numpy()
)

assert set(train_idx).isdisjoint(set(val_idx))

X_model_train = X.iloc[train_idx].copy()
y_model_train = y.iloc[train_idx].copy()

X_val = X.iloc[val_idx].copy()
y_val = y.iloc[val_idx].copy()

split_summary = pd.DataFrame({
    "split": ["model_training", "threshold_validation"],
    "rows": [len(X_model_train), len(X_val)],
    "positive_count": [int(y_model_train.sum()), int(y_val.sum())],
    "positive_rate": [float(y_model_train.mean()), float(y_val.mean())],
})

print(
    "Final-test rows have NOT been loaded. "
    "They stay locked until notebook 17."
)
split_summary


Final-test rows have NOT been loaded. They stay locked until notebook 17.


,split,rows,positive_count,positive_rate
0,model_training,41991,3771,0.089805
1,threshold_validation,13998,1257,0.089799


## 4. Preprocessing

Random Forest does not require feature scaling. Categorical variables are imputed and one-hot encoded. Numeric variables are median-imputed.

In [7]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

random_forest_preprocess = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),
        (
            "numeric",
            numeric_transformer,
            numeric_features
        )
    ],
    remainder="drop"
)

print("Random Forest preprocessing created.")


Random Forest preprocessing created.


## 5. Evaluation and threshold-selection helper functions

In [8]:
def evaluate_predictions_from_proba(
    y_true,
    y_proba,
    threshold=0.5,
    model_name="Model"
):
    """Evaluate binary predictions created from probabilities."""

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    y_pred = (
        y_proba >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    total = tn + fp + fn + tp
    actual_positive = tp + fn
    actual_negative = tn + fp
    predicted_positive = tp + fp
    predicted_negative = tn + fn

    specificity = (
        tn / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_positive_rate = (
        fp / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_negative_rate = (
        fn / actual_positive
        if actual_positive > 0
        else np.nan
    )

    predicted_positive_rate = (
        predicted_positive / total
        if total > 0
        else np.nan
    )

    patients_flagged_per_true_readmission = (
        predicted_positive / tp
        if tp > 0
        else np.nan
    )

    return {
        "model": model_name,
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f2": fbeta_score(
            y_true,
            y_pred,
            beta=2,
            zero_division=0
        ),
        "auroc": roc_auc_score(
            y_true,
            y_proba
        ),
        "auprc": average_precision_score(
            y_true,
            y_proba
        ),
        "brier_score": brier_score_loss(
            y_true,
            y_proba
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "predicted_positive": int(predicted_positive),
        "predicted_negative": int(predicted_negative),
        "predicted_positive_rate": predicted_positive_rate,
        "patients_flagged_per_true_readmission_found": (
            patients_flagged_per_true_readmission
        )
    }


In [9]:
def confusion_matrix_from_proba(
    y_true,
    y_proba,
    threshold=0.5
):
    """Create a labelled confusion matrix from probabilities."""

    y_pred = (
        np.asarray(y_proba) >= threshold
    ).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    return pd.DataFrame(
        cm,
        index=[
            "Actual not readmitted",
            "Actual readmitted"
        ],
        columns=[
            "Predicted not readmitted",
            "Predicted readmitted"
        ]
    )


In [10]:
def threshold_sweep(
    y_true,
    y_proba,
    model_name="Model",
    thresholds=None
):
    """Calculate performance across probability thresholds."""

    if thresholds is None:
        thresholds = np.round(
            np.arange(
                0.01,
                0.951,
                0.01
            ),
            2
        )

    results = [
        evaluate_predictions_from_proba(
            y_true=y_true,
            y_proba=y_proba,
            threshold=threshold,
            model_name=model_name
        )
        for threshold in thresholds
    ]

    return pd.DataFrame(results)


In [11]:
def choose_threshold_for_minimum_recall(
    y_true,
    y_proba,
    min_recall=0.80,
    model_name="Model"
):
    """
    Select the threshold with the lowest false-positive rate
    among thresholds that achieve the required recall.
    """

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    false_positive_rates, recalls, thresholds = roc_curve(
        y_true,
        y_proba,
        drop_intermediate=False
    )

    candidate_table = pd.DataFrame({
        "threshold": thresholds,
        "recall": recalls,
        "false_positive_rate": false_positive_rates,
        "specificity": 1 - false_positive_rates
    })

    candidate_table = candidate_table[
        np.isfinite(
            candidate_table["threshold"]
        )
    ].copy()

    eligible_candidates = candidate_table[
        candidate_table["recall"] >= min_recall
    ].copy()

    if eligible_candidates.empty:
        raise ValueError(
            "No threshold achieved recall >= "
            f"{min_recall:.2f}."
        )

    eligible_candidates = (
        eligible_candidates
        .sort_values(
            by=[
                "false_positive_rate",
                "threshold"
            ],
            ascending=[
                True,
                False
            ]
        )
        .reset_index(drop=True)
    )

    selected_threshold = float(
        eligible_candidates.iloc[0]["threshold"]
    )

    selected_metrics = evaluate_predictions_from_proba(
        y_true=y_true,
        y_proba=y_proba,
        threshold=selected_threshold,
        model_name=model_name
    )

    return (
        selected_threshold,
        selected_metrics,
        eligible_candidates
    )


## 6. Cross-validation settings and hyperparameter search

AUPRC is used for model selection because only about 9% of encounters are positive. Recall is not used directly inside the search because recall depends strongly on the probability threshold. The threshold is selected later on the validation set.

The Random Forest itself uses one CPU during each fit, while RandomizedSearchCV parallelises the separate fits. This avoids nested parallelism and excessive memory use.

In [12]:
RECALL_TARGET = 0.80
SEARCH_ITERATIONS = 60

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(
    f"Required validation recall: "
    f"{RECALL_TARGET:.0%}"
)

print(
    "Cross-validation folds:",
    cross_validation.n_splits
)

print(
    "Random search candidates:",
    SEARCH_ITERATIONS
)


Required validation recall: 80%
Cross-validation folds: 5
Random search candidates: 60


In [13]:
random_forest_param_distributions = {
    "model__n_estimators": [
        200,
        300,
        500,
        700
    ],

    "model__criterion": [
        "gini",
        "entropy",
        "log_loss"
    ],

    "model__max_depth": [
        6,
        8,
        10,
        12,
        16,
        20,
        None
    ],

    "model__min_samples_split": [
        2,
        10,
        25,
        50,
        100,
        200
    ],

    "model__min_samples_leaf": [
        1,
        5,
        10,
        25,
        50,
        100,
        200
    ],

    "model__max_features": [
        "sqrt",
        "log2",
        0.30,
        0.50,
        0.75,
        None
    ],

    "model__max_samples": [
        0.50,
        0.75,
        None
    ],

    "model__class_weight": [
        None,
        "balanced",
        "balanced_subsample",
        {0: 1, 1: 2},
        {0: 1, 1: 4},
        {0: 1, 1: 6},
        {0: 1, 1: 8}
    ]
}


In [14]:
random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocess",
            random_forest_preprocess
        ),
        (
            "model",
            RandomForestClassifier(
                random_state=42,
                bootstrap=True,
                n_jobs=1
            )
        )
    ]
)

random_forest_pipeline


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default

In [15]:
random_forest_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,

    param_distributions=(
        random_forest_param_distributions
    ),

    n_iter=SEARCH_ITERATIONS,

    scoring={
        "auprc": "average_precision",
        "auroc": "roc_auc"
    },

    refit="auprc",

    cv=cross_validation,

    n_jobs=-1,

    verbose=1,

    random_state=42,

    return_train_score=True,

    error_score="raise"
)

random_forest_search.fit(
    X_model_train,
    y_model_train
)


Fitting 5 folds for each of 60 candidates, totalling 300 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__class_weight': [None, 'balanced', ...], 'model__criterion': ['gini', 'entropy', ...], 'model__max_depth': [6, 8, ...], 'model__max_features': ['sqrt', 'log2', ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",60
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","{'auprc': 'average_precision', 'auroc': 'roc_auc'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'auprc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold

## 7. Inspect the selected model and the strongest search candidates

In [16]:
print("Best Random Forest parameters:")

for parameter, value in (
    random_forest_search
    .best_params_
    .items()
):
    print(f"{parameter}: {value}")

print("\nBest cross-validation AUPRC:")
print(random_forest_search.best_score_)


Best Random Forest parameters:
model__n_estimators: 500
model__min_samples_split: 2
model__min_samples_leaf: 5
model__max_samples: None
model__max_features: log2
model__max_depth: 8
model__criterion: gini
model__class_weight: None

Best cross-validation AUPRC:
0.142809011422187


In [17]:
random_forest_cv_results = pd.DataFrame(
    random_forest_search.cv_results_
)

random_forest_cv_results_selected = (
    random_forest_cv_results[
        [
            "rank_test_auprc",
            "mean_test_auprc",
            "std_test_auprc",
            "mean_train_auprc",
            "std_train_auprc",
            "mean_test_auroc",
            "std_test_auroc",
            "param_model__n_estimators",
            "param_model__criterion",
            "param_model__max_depth",
            "param_model__min_samples_split",
            "param_model__min_samples_leaf",
            "param_model__max_features",
            "param_model__max_samples",
            "param_model__class_weight"
        ]
    ]
    .sort_values("rank_test_auprc")
    .reset_index(drop=True)
)

random_forest_cv_results_selected[
    "train_validation_auprc_gap"
] = (
    random_forest_cv_results_selected[
        "mean_train_auprc"
    ]
    -
    random_forest_cv_results_selected[
        "mean_test_auprc"
    ]
)

random_forest_cv_results_selected.to_csv(
    OUTPUT_DIR
    / "random_forest_cv_results.csv",
    index=False
)

random_forest_cv_results_selected.head(20)


,rank_test_auprc,mean_test_auprc,std_test_auprc,mean_train_auprc,std_train_auprc,mean_test_auroc,std_test_auroc,param_model__n_estimators,param_model__criterion,param_model__max_depth,param_model__min_samples_split,param_model__min_samples_leaf,param_model__max_features,param_model__max_samples,param_model__class_weight,train_validation_auprc_gap
0,1,0.142809,0.004842,0.248931,0.003674,0.628515,0.010209,500,gini,8,2,5,log2,None,None,0.106122
1,2,0.142060,0.004526,0.194361,0.002983,0.626130,0.010379,700,gini,6,25,10,None,0.75,None,0.052301
2,3,0.141431,0.006089,0.252876,0.002950,0.627912,0.011651,500,entropy,None,2,50,0.5,0.75,None,0.111445
3,4,0.141061,0.006681,0.225599,0.002389,0.628068,0.011044,700,gini,8,2,25,0.5,0.75,"{0: 1, 1: 4}",0.084538
4,5,0.141051,0.006203,0.261449,0.003861,0.626673,0.011405,500,entropy,None,50,50,0.75,0.75,None,0.120398
5,6,0.141036,0.006028,0.171821,0.001739,0.625822,0.010428,200,log_loss,6,200,25,None,None,"{0: 1, 1: 2}",0.030786
6,7,0.141020,0.005547,0.233971,0.003052,0.626597,0.011193,700,log_loss,16,200,25,0.5,None,balanced,0.092950
7,8,0.140885,0.006265,0.189592,0.001728,0.624318,0.010316,300,gini,6,50,5,0.75,0.75,balanced,0.048707
8,9,0.140377,0.007567,0.217898,0.003926,0.625110,0.011630,300,log_loss,8,50,25,None,0.5,"{0: 1, 1: 8}",0.077521
9,10,0.140366,0.005790,0.171561,0.001252,0.626530,0.011223,300,gini,6,200,50,None,None,balanced_subsample,0.031195


In [18]:
best_random_forest_model = (
    random_forest_search
    .best_estimator_
)

# Parallelise predictions and later calculations after the search.
best_random_forest_model.named_steps[
    "model"
].set_params(n_jobs=-1)

best_random_forest_model


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['gender','race_group','age_group',...,'number_emergency', 'number_inpatient','number_diagnoses']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifyi

In [19]:
fitted_random_forest = (
    best_random_forest_model
    .named_steps["model"]
)

individual_tree_depths = np.array([
    tree.get_depth()
    for tree in fitted_random_forest.estimators_
])

individual_tree_leaves = np.array([
    tree.get_n_leaves()
    for tree in fitted_random_forest.estimators_
])

forest_structure_summary = pd.Series({
    "number_of_trees": len(
        fitted_random_forest.estimators_
    ),
    "minimum_tree_depth": individual_tree_depths.min(),
    "median_tree_depth": np.median(
        individual_tree_depths
    ),
    "mean_tree_depth": individual_tree_depths.mean(),
    "maximum_tree_depth": individual_tree_depths.max(),
    "minimum_leaf_count": individual_tree_leaves.min(),
    "median_leaf_count": np.median(
        individual_tree_leaves
    ),
    "mean_leaf_count": individual_tree_leaves.mean(),
    "maximum_leaf_count": individual_tree_leaves.max()
})

forest_structure_summary.to_csv(
    OUTPUT_DIR
    / "random_forest_structure_summary.csv",
    header=["value"]
)

forest_structure_summary


number_of_trees       500.00
minimum_tree_depth      8.00
median_tree_depth       8.00
mean_tree_depth         8.00
maximum_tree_depth      8.00
minimum_leaf_count     78.00
median_leaf_count     139.00
mean_leaf_count       137.29
maximum_leaf_count    182.00
dtype: float64

## 8. Evaluate the selected model on the validation set at the default threshold

In [20]:
y_val_proba_random_forest = (
    best_random_forest_model
    .predict_proba(X_val)[:, 1]
)

print(
    "Minimum validation probability:",
    y_val_proba_random_forest.min()
)

print(
    "Maximum validation probability:",
    y_val_proba_random_forest.max()
)

print(
    "Number of unique validation probabilities:",
    np.unique(y_val_proba_random_forest).size
)


Minimum validation probability: 0.03631241448002851
Maximum validation probability: 0.2771413606311058
Number of unique validation probabilities: 13997


In [21]:
random_forest_probability_summary = (
    pd.DataFrame({
        "actual_class": np.asarray(y_val),
        "predicted_readmission_probability": (
            y_val_proba_random_forest
        )
    })
    .groupby("actual_class")
    ["predicted_readmission_probability"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

random_forest_probability_summary.to_csv(
    OUTPUT_DIR
    / "random_forest_validation_probability_summary.csv"
)

random_forest_probability_summary


,count,mean,std,min,10%,25%,50%,75%,90%,max
actual_class,,,,,,,,,,
0,12741.0,0.088604,0.027813,0.036312,0.057183,0.066820,0.080695,0.112368,0.125715,0.277141
1,1257.0,0.102662,0.031835,0.038396,0.064449,0.075958,0.103847,0.122486,0.134830,0.267909


In [22]:
random_forest_val_default_results = (
    evaluate_predictions_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_random_forest,
        threshold=0.5,
        model_name=(
            "Random Forest validation default"
        )
    )
)

pd.Series(
    random_forest_val_default_results
)


model                                          Random Forest validation default
threshold                                                                   0.5
accuracy                                                               0.910201
precision                                                                   0.0
recall                                                                      0.0
specificity                                                                 1.0
false_positive_rate                                                         0.0
false_negative_rate                                                         1.0
f1                                                                          0.0
f2                                                                          0.0
auroc                                                                  0.630927
auprc                                                                  0.147898
brier_score                             

## 9. Select the validation threshold that reaches at least 80% recall

Among all validation thresholds that achieve the recall target, this rule selects the one with the lowest false-positive rate.

In [23]:
(
    random_forest_selected_threshold,
    random_forest_val_selected_results,
    random_forest_eligible_thresholds
) = choose_threshold_for_minimum_recall(
    y_true=y_val,
    y_proba=y_val_proba_random_forest,
    min_recall=RECALL_TARGET,
    model_name=(
        "Random Forest validation selected"
    )
)

print("Selected Random Forest threshold:")
print(random_forest_selected_threshold)

pd.DataFrame([
    random_forest_val_default_results,
    random_forest_val_selected_results
])


Selected Random Forest threshold:
0.07262019760531796


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Random Forest validation default,0.50000,0.910201,0.000000,0.000000,1.000000,0.000000,1.000000,0.00000,0.000000,...,0.147898,0.080248,12741,0,1257,0,0,13998,0.000000,NaN
1,Random Forest validation selected,0.07262,0.407201,0.111123,0.800318,0.368417,0.631583,0.199682,0.19515,0.357219,...,0.147898,0.080248,4694,8047,251,1006,9053,4945,0.646735,8.999006


In [24]:
random_forest_eligible_thresholds.to_csv(
    OUTPUT_DIR
    / "random_forest_eligible_validation_thresholds.csv",
    index=False
)

random_forest_eligible_thresholds.head(20)


,threshold,recall,false_positive_rate,specificity
0,0.072620,0.800318,0.631583,0.368417
1,0.072619,0.800318,0.631662,0.368338
2,0.072615,0.800318,0.631740,0.368260
3,0.072611,0.800318,0.631819,0.368181
4,0.072610,0.800318,0.631897,0.368103
5,0.072609,0.800318,0.631976,0.368024
6,0.072608,0.800318,0.632054,0.367946
7,0.072607,0.800318,0.632132,0.367868
8,0.072604,0.800318,0.632211,0.367789
9,0.072602,0.800318,0.632289,0.367711


In [25]:
random_forest_threshold_sweep = threshold_sweep(
    y_true=y_val,
    y_proba=y_val_proba_random_forest,
    model_name="Random Forest validation"
)

random_forest_threshold_sweep.to_csv(
    OUTPUT_DIR
    / "random_forest_validation_threshold_sweep.csv",
    index=False
)

random_forest_threshold_sweep[
    [
        "threshold",
        "recall",
        "precision",
        "specificity",
        "false_positive_rate",
        "f1",
        "f2",
        "true_positive",
        "false_positive",
        "false_negative",
        "predicted_positive_rate"
    ]
]


,threshold,recall,precision,specificity,false_positive_rate,f1,f2,true_positive,false_positive,false_negative,predicted_positive_rate
0,0.01,1.000000,0.089799,0.000000,1.000000,0.164798,0.330337,1257,12741,0,1.000000
1,0.02,1.000000,0.089799,0.000000,1.000000,0.164798,0.330337,1257,12741,0,1.000000
2,0.03,1.000000,0.089799,0.000000,1.000000,0.164798,0.330337,1257,12741,0,1.000000
3,0.04,0.999204,0.089881,0.001805,0.998195,0.164927,0.330492,1256,12718,1,0.998285
4,0.05,0.993636,0.092123,0.033906,0.966094,0.168613,0.336006,1249,12309,8,0.968567
...,...,...,...,...,...,...,...,...,...,...,...
90,0.91,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0,0,1257,0.000000
91,0.92,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0,0,1257,0.000000
92,0.93,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0,0,1257,0.000000
93,0.94,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0,0,1257,0.000000


In [26]:
comparison_columns = [
    "model",
    "threshold",
    "auprc",
    "auroc",
    "brier_score",
    "accuracy",
    "recall",
    "precision",
    "specificity",
    "false_positive_rate",
    "false_negative_rate",
    "f1",
    "f2",
    "true_positive",
    "true_negative",
    "false_positive",
    "false_negative",
    "predicted_positive_rate",
    "patients_flagged_per_true_readmission_found"
]

random_forest_validation_comparison = pd.DataFrame([
    random_forest_val_default_results,
    random_forest_val_selected_results
])

random_forest_validation_comparison = (
    random_forest_validation_comparison[
        comparison_columns
    ]
)

random_forest_validation_comparison.to_csv(
    OUTPUT_DIR
    / "random_forest_validation_results.csv",
    index=False
)

random_forest_validation_comparison


,model,threshold,auprc,auroc,brier_score,accuracy,recall,precision,specificity,false_positive_rate,false_negative_rate,f1,f2,true_positive,true_negative,false_positive,false_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Random Forest validation default,0.50000,0.147898,0.630927,0.080248,0.910201,0.000000,0.000000,1.000000,0.000000,1.000000,0.00000,0.000000,0,12741,0,1257,0.000000,NaN
1,Random Forest validation selected,0.07262,0.147898,0.630927,0.080248,0.407201,0.800318,0.111123,0.368417,0.631583,0.199682,0.19515,0.357219,1006,4694,8047,251,0.646735,8.999006


In [27]:
random_forest_val_selected_cm = (
    confusion_matrix_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_random_forest,
        threshold=(
            random_forest_selected_threshold
        )
    )
)

random_forest_val_selected_cm.to_csv(
    OUTPUT_DIR
    / "random_forest_validation_confusion_matrix.csv"
)

random_forest_val_selected_cm


,Predicted not readmitted,Predicted readmitted
Actual not readmitted,4694,8047
Actual readmitted,251,1006


In [28]:
y_val_pred_random_forest = (
    y_val_proba_random_forest
    >= random_forest_selected_threshold
).astype(int)

print(
    classification_report(
        y_val,
        y_val_pred_random_forest,
        target_names=[
            "Not readmitted",
            "Readmitted"
        ],
        zero_division=0
    )
)


                precision    recall  f1-score   support

Not readmitted       0.95      0.37      0.53     12741
    Readmitted       0.11      0.80      0.20      1257

      accuracy                           0.41     13998
     macro avg       0.53      0.58      0.36     13998
  weighted avg       0.87      0.41      0.50     13998



## Freeze final development-stage artifacts

In [29]:

RECALL_TARGETS = [0.70, 0.75, 0.80, 0.85, 0.90]

threshold_rows = []

for recall_target in RECALL_TARGETS:
    selected_threshold, selected_metrics, _ = (
        choose_threshold_for_minimum_recall(
            y_true=y_val,
            y_proba=y_val_proba_random_forest,
            min_recall=recall_target,
            model_name="Random Forest validation"
        )
    )

    threshold_rows.append({
        "target_recall": recall_target,
        "selected_threshold": selected_threshold,
        "validation_recall": selected_metrics["recall"],
        "validation_precision": selected_metrics["precision"],
        "validation_specificity": selected_metrics["specificity"],
        "validation_false_positive_rate": selected_metrics["false_positive_rate"],
        "validation_f2": selected_metrics["f2"],
        "validation_true_positive": selected_metrics["true_positive"],
        "validation_false_negative": selected_metrics["false_negative"],
        "validation_false_positive": selected_metrics["false_positive"],
        "validation_true_negative": selected_metrics["true_negative"],
        "validation_flagged_rate": selected_metrics["predicted_positive_rate"],
    })

selected_thresholds = pd.DataFrame(threshold_rows)

selected_thresholds.to_csv(
    OUTPUT_DIR / "selected_validation_thresholds.csv",
    index=False
)

validation_predictions = pd.DataFrame({
    "row_position": val_idx,
    "y_true": np.asarray(y_val, dtype=int),
    "probability": np.asarray(y_val_proba_random_forest, dtype=float),
})

validation_predictions.to_csv(
    OUTPUT_DIR / "validation_predictions.csv",
    index=False
)

selected_thresholds

import joblib
joblib.dump(best_random_forest_model, OUTPUT_DIR / "final_model.joblib")

with open(OUTPUT_DIR / "selected_hyperparameters.json", "w") as f:
    json.dump(random_forest_search.best_params_, f, indent=2, default=str)

print("Saved frozen Random Forest model and artifacts.")


Saved frozen Random Forest model and artifacts.
